# 15 — Version 2 LightGBM + CatBoost consensus

**Owners:** Midhun / Saravana / Nebal / Ajmeer

Run only after notebooks 11 and 12 have complete full-data runs. This notebook
chooses each model by validation PR-AUC, learns one LightGBM/CatBoost log-odds
weight on validation, freezes it, and evaluates the test period once.

The four original outputs remain visible. Consensus is an additional fifth score.


In [ ]:
from pathlib import Path
_start = Path.cwd().resolve()
for _candidate in [_start, *_start.parents]:
    if (_candidate / "requirements-training.txt").exists():
        _requirements = _candidate / "requirements-training.txt"
        break
else:
    raise FileNotFoundError("Open this notebook from inside the cloned repository")
%pip install -q -r {_requirements}


In [ ]:
from pathlib import Path
import gc, json, os, sys, time
import numpy as np
import pandas as pd

def locate_project_root(start=None):
    candidate = Path(start or Path.cwd()).resolve()
    for path in [candidate, *candidate.parents]:
        if (path / ".git").exists() and (path / "src").exists():
            return path
    raise FileNotFoundError("Run this notebook from inside the cloned repository")

PROJECT_ROOT = locate_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

V2_DATA_DIR = PROJECT_ROOT / "data" / "processed" / "v2"
V2_ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "v2"
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print("Project root:", PROJECT_ROOT)
print("Version 2 data:", V2_DATA_DIR)
print("Version 2 artifacts:", V2_ARTIFACT_ROOT)


In [ ]:
MODEL_KEY = "consensus"

from datetime import datetime, timezone
from src.fraud_pipeline.artifacts import build_manifest, package_versions, write_json
from src.fraud_pipeline.evaluation import evaluate_binary_classifier, select_operating_threshold

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = V2_ARTIFACT_ROOT / MODEL_KEY / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("This Version 2 run will be saved to:", RUN_DIR)


In [ ]:
from sklearn.metrics import average_precision_score
from src.fraud_pipeline.validation_v2 import (
    apply_two_model_logit_blend, best_complete_run,
    fit_two_model_logit_blend, merge_prediction_files,
)

source_runs = {
    "lightgbm": best_complete_run(V2_ARTIFACT_ROOT, "lightgbm"),
    "catboost": best_complete_run(V2_ARTIFACT_ROOT, "catboost"),
}
print({name: path.name for name, path in source_runs.items()})
validation_predictions = merge_prediction_files(source_runs, "validation")
test_predictions = merge_prediction_files(source_runs, "test")
blend = fit_two_model_logit_blend(
    validation_predictions.isFraud.to_numpy(),
    validation_predictions.lightgbm.to_numpy(),
    validation_predictions.catboost.to_numpy(),
)
print("Validation-selected blend:", blend)


## Freeze the blend and evaluate


In [ ]:
validation_probability = apply_two_model_logit_blend(
    validation_predictions.lightgbm, validation_predictions.catboost,
    blend["first_weight"])
threshold_record = select_operating_threshold(
    validation_predictions.isFraud.to_numpy(), validation_probability,
    minimum_precision=0.10)
threshold = float(threshold_record["threshold"])
validation_metrics = evaluate_binary_classifier(
    validation_predictions.isFraud.to_numpy(), validation_probability, threshold)
test_probability = apply_two_model_logit_blend(
    test_predictions.lightgbm, test_predictions.catboost, blend["first_weight"])
test_metrics = evaluate_binary_classifier(
    test_predictions.isFraud.to_numpy(), test_probability, threshold)
display(pd.DataFrame([validation_metrics, test_metrics], index=["validation", "test"])[
    ["pr_auc", "roc_auc", "precision", "recall", "f1", "brier_score"]])


## Save the deployable ensemble recipe


In [ ]:
pd.DataFrame({"TransactionID": validation_predictions.TransactionID,
    "isFraud": validation_predictions.isFraud,
    "probability": validation_probability}).to_parquet(
        RUN_DIR / "validation_predictions.parquet", index=False)
pd.DataFrame({"TransactionID": test_predictions.TransactionID,
    "isFraud": test_predictions.isFraud,
    "probability": test_probability}).to_parquet(
        RUN_DIR / "test_predictions.parquet", index=False)
write_json(RUN_DIR / "threshold.json", threshold_record)
write_json(RUN_DIR / "metrics.json", {"validation": validation_metrics, "test": test_metrics})
write_json(RUN_DIR / "consensus_config.json", {
    "method": "weighted_log_odds", "first_model": "lightgbm",
    "second_model": "catboost", "first_weight": blend["first_weight"],
    "second_weight": blend["second_weight"],
    "source_runs": {name: path.name for name, path in source_runs.items()},
})
write_json(RUN_DIR / "training_config.json", {
    "model": "v2_consensus", "run_id": RUN_ID, "fast_run": False,
    "selection_data": "validation_only", "test_used_for_weight_selection": False,
    "versions": package_versions(["numpy", "pandas", "scikit-learn"]),
})


In [ ]:
import shutil
write_json(RUN_DIR / "manifest.json", build_manifest(RUN_DIR))
archive_base = RUN_DIR.parent / f"{MODEL_KEY}_{RUN_ID}"
archive_path = Path(shutil.make_archive(str(archive_base), "gztar", root_dir=RUN_DIR))
print("Reload check passed.")
print("Artifact folder:", RUN_DIR)
print("Share this archive:", archive_path)
